In [38]:
from langchain import agents
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

True

In [39]:
model=init_chat_model(model="groq:openai/gpt-oss-120b")

In [40]:
@tool
def send_email():
    """ you will generate a email...for need help in landslide to the authority including the location and date"""
    location="kolkata"
    prompt=PromptTemplate(
        input_variables=["location"],
        template="i will provide you a loaction :{location} you will generate a email for asking help in land slide"
    )
    chain=prompt | model | StrOutputParser()
    print(f"creating email:\n\n{chain.invoke({"location":location})}\n\n")


In [41]:
@tool
def call_in_emergency_number():
    """you will autamtically call in emergency number store by user"""
    print("calling to 9111")

@tool
def normal_chat():
    """if not emergency just respose to normal questions"""
    print("normal chat")

In [42]:
@tool
def send_email(location: str = "kolkata"):
    """Generate an email for landslide help request."""
    prompt = PromptTemplate(
        input_variables=["location"],
        template="I will provide you a location: {location}. Generate an email asking for help in a landslide."
    )
    chain = prompt | model | StrOutputParser()
    email_text = chain.invoke({"location": location})
    return {"action": "send_email", "email": email_text}

@tool
def call_in_emergency_number():
    """Automatically call the emergency number stored by user."""
    return {"action": "call_emergency", "number": "9111"}

@tool
def normal_chat(message: str = "Hello"):
    """Respond to normal questions."""
    return {"action": "chat", "response": f"Normal chat: {message}"}


In [46]:
agent = create_agent(
    model=model,
    tools=[send_email, call_in_emergency_number, normal_chat],
   system_prompt = """
You are a chat model. 
If the user mentions emergencies like flood, landslide, earthquake, accident, danger, or being stuck, 
call the emergency tools. 
Otherwise, use normal chat.
If the user mentions emergencies like flood, landslide, earthquake, accident, or danger:
- Call the emergency number tool
- Also generate an email to the authorities with location and date

User: "I am stuck in a flood"
Assistant: {tool: "call_in_emergency_number"}, {tool: "send_email"}


"""

)

result = agent.invoke({"input":"its an flood"})
print(result)


{'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says "I am stuck in a flood". This is an emergency. According to developer instructions, we need to call the emergency number tool and also generate an email to the authorities with location and date. The email tool defined is send_email with optional location default "kolkata". We need location - not provided. Could ask? Probably use default. Also need date? The email tool only takes location. There\'s no date param. So we just call send_email with default location. Also call call_in_emergency_number with empty args.\n\nThus we need to output two tool calls.', 'tool_calls': [{'id': 'fc_ce91f4b9-c593-4f40-aa39-4f9952d3c8de', 'function': {'arguments': '{}', 'name': 'call_in_emergency_number'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 286, 'total_tokens': 426, 'completion_time': 0.294878524, 'completion_tokens_details': {'reasoning_tokens': 118

In [ ]:
# result is the dict returned by agent.invoke(...)
for msg in result["messages"]:
    if msg.__class__.__name__ == "ToolMessage" and msg.name == "send_email":
        email_data = msg.content  # JSON string
        print(email_data)


ValueError: dictionary update sequence element #0 has length 1; 2 is required